# Day 29: GenAI System Evaluation
**Relevance and faithfulness metrics, RAG evaluation, and bias and hallucination analysis**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. The system under test: a small RAG assistant

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
import re, numpy as np, pandas as pd
import genai_helpers

text = open("knowledge.md").read()
CHUNKS = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", text)) if s.strip()]
CVECS = embed(CHUNKS)
RULES = "Answer using ONLY the context. If the answer is not there, say \"I don't know\"."

def retrieve(q, k=3):
    sims = CVECS @ embed(q)[0]
    order = np.argsort(sims)[::-1][:k]
    return [CHUNKS[i] for i in order], [float(sims[i]) for i in order]

def good_generator(q, ctx):
    return llm("Context:\n" + "\n".join(ctx) + f"\n\nQuestion: {q}", system=RULES, temperature=0)

def hallucinating_generator(q, ctx):
    return good_generator(q, ctx) + " Employees also get a free gym membership and a yearly bonus."

print(len(CHUNKS), "chunks indexed")

## 3. The evaluation set
Realistic wording, expected key phrase, source sentence, topic and language. Includes Japanese questions and unanswerable ones.

In [ ]:
EVAL = [
    {"q": "How many sick days do I get each year?", "expect": "8 days", "source": "8 days of paid sick leave", "topic": "leave", "lang": "en"},
    {"q": "When does leave I carried over expire?", "expect": "31 March", "source": "expires on 31 March", "topic": "leave", "lang": "en"},
    {"q": "How many days can I work from home each week?", "expect": "2 days", "source": "up to 2 days per week", "topic": "remote", "lang": "en"},
    {"q": "What does error VPN-417 mean?", "expect": "certificate has expired", "source": "VPN-417", "topic": "it", "lang": "en"},
    {"q": "How long must my password be?", "expect": "14 characters", "source": "14 characters", "topic": "it", "lang": "en"},
    {"q": "What is the hotel limit per night in Japan?", "expect": "15,000 JPY", "source": "15,000 JPY", "topic": "travel", "lang": "en"},
    {"q": "Within how many days must travel expenses be submitted?", "expect": "30 days", "source": "within 30 days", "topic": "travel", "lang": "en"},
    {"q": "How quickly must security incidents be reported?", "expect": "1 hour", "source": "within 1 hour", "topic": "security", "lang": "en"},
    {"q": "病気休暇は年に何日ですか？", "expect": "8 days", "source": "8 days of paid sick leave", "topic": "leave", "lang": "ja"},
    {"q": "日本のホテルの上限はいくらですか？", "expect": "15,000 JPY", "source": "15,000 JPY", "topic": "travel", "lang": "ja"},
    {"q": "パスワードは何文字必要ですか？", "expect": "14 characters", "source": "14 characters", "topic": "it", "lang": "ja"},
    {"q": "Is there a free company gym?", "expect": "don't know", "source": None, "topic": "other", "lang": "en"},
]
print(len(EVAL), "items |", pd.Series([e["lang"] for e in EVAL]).value_counts().to_dict())

## 4. Metrics

In [ ]:
WORDS = lambda t: {w for w in re.findall(r"[a-z0-9,]+", t.lower()) if len(w) > 3}

def faithfulness(answer, ctx):
    claims = [c for c in re.split(r"(?<=[.!?])\s+", answer) if c.strip()]
    if not claims or "don't know" in answer.lower():
        return 1.0
    context = " ".join(ctx)
    def supported(claim):
        if ONLINE:
            r = llm(f"Context: {context}\nClaim: {claim}\nIs the claim fully supported by the context? Reply yes or no.", temperature=0)
            return r.strip().lower().startswith("yes")
        w = WORDS(claim)
        return bool(w) and len(w & WORDS(context)) / len(w) >= 0.6     # offline stand-in
    return sum(supported(c) for c in claims) / len(claims)

def evaluate(generator):
    rows = []
    for e in EVAL:
        ctx, sims = retrieve(e["q"])
        rank = next((i + 1 for i, c in enumerate(ctx) if e["source"] and e["source"] in c), None)
        ans = generator(e["q"], ctx)
        correct = e["expect"].lower() in ans.lower()
        rows.append({"q": e["q"][:38], "lang": e["lang"], "topic": e["topic"],
                     "hit@3": rank is not None if e["source"] else None,
                     "rr": (1 / rank if rank else 0.0) if e["source"] else None,
                     "context_rel": round(float(np.mean(sims)), 3),
                     "answer_rel": round(float(embed(e["q"])[0] @ embed(ans)[0]), 3),
                     "faithful": round(faithfulness(ans, ctx), 2),
                     "correct": correct,
                     "error": "ok" if correct else ("retrieval_miss" if e["source"] and rank is None else "generation_error"),
                     "answer": ans[:70]})
    return pd.DataFrame(rows)

good = evaluate(good_generator)
print(good[["q", "lang", "hit@3", "faithful", "correct", "error"]].to_string(index=False))

## 5. Summary scores, and catching a planted hallucination

In [ ]:
bad = evaluate(hallucinating_generator)
summary = pd.DataFrame({
    "good generator": [good["hit@3"].dropna().mean(), good["rr"].dropna().mean(), good["faithful"].mean(), good["correct"].mean()],
    "hallucinating": [bad["hit@3"].dropna().mean(), bad["rr"].dropna().mean(), bad["faithful"].mean(), bad["correct"].mean()],
}, index=["hit@3", "MRR", "faithfulness", "correctness"]).round(2)
print(summary)
print("\nRetrieval scores are identical; faithfulness drops. That is how you tell a generation problem from a retrieval problem.")

## 6. Error analysis and slices (bias check)

In [ ]:
print(good["error"].value_counts().to_string(), "\n")
print("Correctness by language:\n", good.groupby("lang")["correct"].mean().round(2).to_string(), "\n")
print("Correctness by topic:\n", good.groupby("topic")["correct"].mean().round(2).to_string())
if not ONLINE:
    print("\nOffline embeddings only match shared words, so Japanese questions cannot find English text. "
          "This is a real class of bias: always slice by language. With a multilingual embedding model the gap should shrink; measure it.")

## 7. LLM as judge with a rubric (needs a key for real scores)

In [ ]:
RUBRIC = '''Score the ANSWER for correctness against the EXPECTED answer.
5 = fully correct, 3 = partly correct or vague, 1 = wrong or missing.
First write one sentence of reasoning, then on a new line: Score: <1-5>'''

def judge(q, expected, answer):
    if ONLINE:
        r = llm(f"QUESTION: {q}\nEXPECTED: {expected}\nANSWER: {answer}", system=RUBRIC, temperature=0)
        m = re.search(r"Score:\s*(\d)", r)
        return int(m.group(1)) if m else None, r.splitlines()[0]
    return (5 if expected.lower() in answer.lower() else 1), "offline: exact phrase match stand-in"

for e, ans in list(zip(EVAL, good["answer"]))[:4]:
    s, why = judge(e["q"], e["expect"], ans)
    print(f"{s} | {e['q'][:40]} | {why}")

### TODO 1: Label the first 8 answers yourself (1 to 5) and measure how often the judge agrees within 1 point.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)